In [1]:

import os
import math
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torch.amp import GradScaler, autocast


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")


train_path = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/annotations_v2/SI/train.txt"
dev_path   = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/annotations_v2/SI/dev.txt"
pkl_path   = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/data/pose_data_isharah1000_hands_lips_body_May12.pkl"


Device: cuda


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:

df_train = pd.read_csv(train_path, sep="|")
df_dev   = pd.read_csv(dev_path,   sep="|")

print(df_train.head())
print(df_dev.head())
print("Train samples:", len(df_train))
print("Dev samples:", len(df_dev))

with open(pkl_path, "rb") as f:
    pose_dict = pickle.load(f)

print("Total sequences in PKL:", len(pose_dict))
sample_key = next(iter(pose_dict.keys()))
print("Example sample shape:", pose_dict[sample_key]["keypoints"].shape)


        id                 gloss                   text
0  00_0001               سوال هو                  من هو
1  00_0002     هو معلم لغه اشاره      هو مدرس لغه اشاره
2  00_0003    استفهام هو معلم هو            هل انت مدرس
3  00_0004  هو معلم لا انا مدرسه  انت لست مدرس انا طالب
4  00_0005               هو سوال                  هو من
        id                 gloss
0  02_0001               سوال هو
1  02_0002     هو معلم لغه اشاره
2  02_0003    استفهام هو معلم هو
3  02_0004  هو معلم لا انا مدرسه
4  02_0005               هو سوال
Train samples: 10000
Dev samples: 949


/tmp/ipython-input-3384138759.py:12: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_dict = pickle.load(f)


Total sequences in PKL: 10450
Example sample shape: (55, 86, 2)


In [3]:

# Collect all gloss tokens
all_gloss_tokens = []

for g in df_train["gloss"]:
    all_gloss_tokens.extend(g.split())

for g in df_dev["gloss"]:
    all_gloss_tokens.extend(g.split())

vocab = ["<BLANK>"] + sorted(set(all_gloss_tokens))

g2i = {g: i for i, g in enumerate(vocab)}
i2g = {i: g for g, i in g2i.items()}

print("Vocab size:", len(vocab))
print("Example tokens:", vocab[:20])

def encode_gloss_seq(sentence: str):
    return [g2i[token] for token in sentence.split()]

df_train["encoded"] = df_train["gloss"].apply(encode_gloss_seq)
df_dev["encoded"]   = df_dev["gloss"].apply(encode_gloss_seq)

df_train[["id","encoded"]].head()


Vocab size: 684
Example tokens: ['<BLANK>', 'ا', 'اب', 'ابتسامه', 'ابن', 'ابها', 'ابيض', 'اتصال', 'اتصالات', 'اتقان', 'اثنان', 'اثنان_ثلاثون', 'اثنين', 'اجازه', 'اجتماع', 'احد_عشر', 'احرام', 'احمر', 'اخ', 'اخبار']


,id,encoded
0,00_0001,"[358, 661]"
1,00_0002,"[661, 607, 555, 54]"
2,00_0003,"[45, 661, 607, 661]"
3,00_0004,"[661, 607, 548, 98, 580]"
4,00_0005,"[661, 358]"


In [4]:

def load_pose(sample_id):
    if sample_id not in pose_dict:
        return None
    return pose_dict[sample_id]["keypoints"]   # (T, 86, 2)


def normalize_pose(pose: np.ndarray):
    """
    pose: (T, 86, 2) in pixel coords.
    Normalize per sequence to [-1, 1].
    """
    pose = pose.astype(np.float32)

    # Center around first joint (frame-wise)
    ref = pose[:, :1, :]        # (T,1,2)
    pose = pose - ref

    # Global scaling to [-1, 1]
    min_val = pose.min()
    max_val = pose.max()
    scale = max_val - min_val
    if scale < 1e-6:
        return np.zeros_like(pose, dtype=np.float32)

    pose = (pose - min_val) / (scale + 1e-6)  # [0,1]
    pose = pose * 2.0 - 1.0                   # [-1,1]
    return pose


X_train, Y_train = [], []
missing_train = []

for idx, row in df_train.iterrows():
    sample_id = row["id"]
    pose = load_pose(sample_id)

    if pose is None:
        missing_train.append(sample_id)
        continue

    pose = normalize_pose(pose)
    X_train.append(pose)           # (T, 86, 2)
    Y_train.append(row["encoded"]) # list[int]

print("Loaded train:", len(X_train))
print("Missing train (first 10):", missing_train[:10])

X_dev, Y_dev = [], []
missing_dev = []

for idx, row in df_dev.iterrows():
    sample_id = row["id"]
    pose = load_pose(sample_id)

    if pose is None:
        missing_dev.append(sample_id)
        continue

    pose = normalize_pose(pose)
    X_dev.append(pose)
    Y_dev.append(row["encoded"])

print("Loaded dev:", len(X_dev))
print("Missing dev (first 10):", missing_dev[:10])

print("Final X_train:", len(X_train))
print("Final X_dev:", len(X_dev))
print("Final Y_train:", len(Y_train))
print("Final Y_dev:", len(Y_dev))
print("Vocabulary size:", len(vocab))


Loaded train: 9500
Missing train (first 10): ['00_0451', '00_0452', '00_0453', '00_0454', '00_0455', '00_0456', '00_0457', '00_0458', '00_0459', '00_0460']
Loaded dev: 949
Missing dev (first 10): []
Final X_train: 9500
Final X_dev: 949
Final Y_train: 9500
Final Y_dev: 949
Vocabulary size: 684


In [5]:

class SignDataset(Dataset):
    def __init__(self, X, Y, max_len=300):
        """
        X: list of np.array (T, 86, 2)
        Y: list of list[int]
        max_len: pad/crop length in frames
        """
        self.X = X
        self.Y = Y
        self.max_len = max_len

    def __len__(self):
        return len(self.X)

    def pad_seq(self, pose_tensor: torch.Tensor):
        """
        pose_tensor: (T, 86, 2)
        returns:
          pose_flat: (max_len, 172)
          T_orig: original T before pad/crop
        """
        T, J, D = pose_tensor.shape
        T_orig = min(T, self.max_len)

        if T > self.max_len:
            pose_tensor = pose_tensor[:self.max_len]
        else:
            pad_len = self.max_len - T
            pad = torch.zeros((pad_len, J, D), dtype=torch.float32)
            pose_tensor = torch.cat([pose_tensor, pad], dim=0)

        pose_flat = pose_tensor.view(self.max_len, -1)  # (max_len, 172)
        return pose_flat, T_orig

    def __getitem__(self, idx):
        pose_np = self.X[idx]                  # (T, 86, 2)
        label   = self.Y[idx]                  # list[int]

        pose = torch.tensor(pose_np, dtype=torch.float32)
        label_t = torch.tensor(label, dtype=torch.long)

        pose_flat, T_orig = self.pad_seq(pose)
        return pose_flat, label_t, T_orig      # (T,172), (L,), scalar


BLANK_ID = 0

def ctc_collate_fn(batch):
    poses = []
    labels = []
    input_lengths = []
    target_lengths = []

    for pose, label, T_orig in batch:
        poses.append(pose)              # (T,172)
        labels.append(label)            # (L,)
        input_lengths.append(T_orig)
        target_lengths.append(len(label))

    poses = torch.stack(poses, dim=0)   # (B, T, 172)
    targets = torch.cat(labels, dim=0)  # (sum_L,)

    return (
        poses,                                      # (B, T, 172)
        targets,                                    # (sum_L,)
        torch.tensor(input_lengths, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
        labels                                      # keep for WER
    )


batch_size = 32

train_ds = SignDataset(X_train, Y_train, max_len=300)
dev_ds   = SignDataset(X_dev,   Y_dev,   max_len=300)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=4
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=4
)

print("Train batches:", len(train_loader))
print("Dev batches:", len(dev_loader))


Train batches: 297
Dev batches: 30


In [6]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=300):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0), persistent=True)  # (1, T, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        T = x.size(1)
        return x + self.pe[:, :T, :]


class PoseTransformerCTC(nn.Module):
    def __init__(
        self,
        input_dim=172,
        d_model=384,
        nheads=6,
        num_layers=6,
        num_classes=None
    ):
        super().__init__()
        assert num_classes is not None, "num_classes must be provided"

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=300)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nheads,
            dim_feedforward=1536,
            dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(0.1)
        self.fc_out = nn.Linear(d_model, num_classes)

    def forward(self, x, src_key_padding_mask=None):
        """
        x: (B, T, 172)
        src_key_padding_mask: (B, T) bool, True for PAD positions
        """
        x = self.input_proj(x)                         # (B, T, d_model)
        x = self.pos_enc(x)                            # (B, T, d_model)
        x = self.encoder(
            x,
            src_key_padding_mask=src_key_padding_mask  # (B, T)
        )
        x = self.dropout(x)
        logits = self.fc_out(x)                        # (B, T, num_classes)
        return logits


num_classes = len(vocab)
model = PoseTransformerCTC(num_classes=num_classes).to(device)

ctc_loss = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

scaler = GradScaler("cuda") if device.type == "cuda" else None

print(model)


PoseTransformerCTC(
  (input_proj): Linear(in_features=172, out_features=384, bias=True)
  (pos_enc): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
        )
        (linear1): Linear(in_features=384, out_features=1536, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1536, out_features=384, bias=True)
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc_out): Linear(in_features=384, out_features=684, bias=True)
)


In [7]:

def greedy_decode_ctc(logits, input_lengths=None, blank_id=BLANK_ID):
    """
    logits: (B, T, C)
    input_lengths: (B,) tensor with true lengths (optional)
    returns: list[list[int]] per sample
    """
    B, T, C = logits.shape
    pred_ids = logits.argmax(dim=-1).cpu()  # (B, T)
    outputs = []

    for b in range(B):
        if input_lengths is not None:
            L = int(input_lengths[b].item())
            seq = pred_ids[b, :L].tolist()
        else:
            seq = pred_ids[b].tolist()

        prev = None
        decoded = []
        for idx in seq:
            if idx == blank_id:
                prev = None
                continue
            if idx != prev:
                decoded.append(idx)
                prev = idx
        outputs.append(decoded)

    return outputs


def edit_distance(ref, hyp):
    n = len(ref)
    m = len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],    # deletion
                    dp[i][j - 1],    # insertion
                    dp[i - 1][j - 1] # substitution
                )
    return dp[n][m]


def compute_wer(refs, hyps):
    """
    refs, hyps: list[list[int]] (label indices)
    """
    total_words = 0
    total_errs = 0
    for ref, hyp in zip(refs, hyps):
        ref_gloss = [i2g[i] for i in ref]
        hyp_gloss = [i2g[i] for i in hyp]
        total_words += len(ref_gloss)
        total_errs += edit_distance(ref_gloss, hyp_gloss)
    if total_words == 0:
        return 0.0
    return total_errs / total_words


In [8]:

num_epochs = 100

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    for batch_x, targets, input_lengths, target_lengths, _ in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)   # (B, T, 172)
        targets = targets.to(device, non_blocking=True)
        input_lengths = input_lengths.to(device)
        target_lengths = target_lengths.to(device)

        B, T, _ = batch_x.shape
        # True for padding positions
        mask = torch.arange(T, device=device).unsqueeze(0) >= input_lengths.unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)

        if device.type == "cuda":
            with autocast("cuda"):
                logits = model(batch_x, src_key_padding_mask=mask)   # (B, T, C)
                log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)  # (T, B, C)
                loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(batch_x, src_key_padding_mask=mask)
            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    refs = []
    hyps = []

    with torch.no_grad():
        for batch_x, _, input_lengths, target_lengths, labels in dev_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            input_lengths = input_lengths.to(device)

            B, T, _ = batch_x.shape
            mask = torch.arange(T, device=device).unsqueeze(0) >= input_lengths.unsqueeze(1)

            logits = model(batch_x, src_key_padding_mask=mask)   # (B, T, C)

            pred_ids = greedy_decode_ctc(logits, input_lengths)  # list[list[int]]

            hyps.extend(pred_ids)
            refs.extend([lbl.tolist() for lbl in labels])

        wer = compute_wer(refs, hyps)

    print(f"Epoch {epoch:02d}: Train Loss={avg_loss:.4f} | Dev WER={wer:.4f}")

    if epoch == 1 or epoch % 5 == 0:
        print("Sample GT glosses:   ", [i2g[i] for i in refs[0]])
        print("Sample Pred glosses: ", [i2g[i] for i in hyps[0]])
        print("-" * 60)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01: Train Loss=15.2282 | Dev WER=1.0000
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 02: Train Loss=5.9133 | Dev WER=1.0000
Epoch 03: Train Loss=5.7866 | Dev WER=1.0000
Epoch 04: Train Loss=5.7099 | Dev WER=0.9194
Epoch 05: Train Loss=5.6322 | Dev WER=0.9194
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 06: Train Loss=5.5627 | Dev WER=0.9194
Epoch 07: Train Loss=5.5001 | Dev WER=0.9192
Epoch 08: Train Loss=5.4408 | Dev WER=0.9137
Epoch 09: Train Loss=5.3751 | Dev WER=0.9093
Epoch 10: Train Loss=5.2859 | Dev WER=0.9091
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 11: Train Loss=5.1903 | Dev WER=0.9045
Epoch 12: Train Loss=5.0877 | Dev WER=0.9007
Epoch 13: Train Loss=4.9704 | Dev WER=0.8966
Epoch 14: Train Loss=4.8252 | Dev WER=0.8909


Model Description

Transformer-Based Continuous Sign Language Recognition (CSLR)

We implemented a Transformer-based Continuous Sign Language Recognition (CSLR) model that operates directly on 2D pose keypoints extracted from the Isharah dataset. Each input video is represented as a sequence of (T, 86, 2) keypoints encompassing the body skeleton, hands, face, and lips.

1- Preprocessing

Each pose sequence is first normalized:

Centered around the reference joint (first joint)

Scaled to the range [-1, 1] across the entire sequence

Flattened frame-wise into a 172-dimensional vector

All sequences are then padded or cropped to a fixed length of 300 frames, and a padding mask is created to ensure the Transformer does not attend to padded regions.

2- Input Projection & Positional Encoding

Each frame vector is projected to a 384-dimensional embedding using a linear layer.
To encode temporal order, we apply sinusoidal positional encodings which are added to the projected features.

3- Transformer Encoder

The core of the model is a 6-layer Transformer encoder with:

Hidden size: 384

Multi-head self-attention with 6 heads

Feed-forward dimension: 1536

Dropout: 0.1

Self-attention enables the model to capture long-range temporal dependencies and interactions between motion patterns across different joints. A padding mask ensures that the model only attends to valid frames.

4- Classification Layer

The encoder outputs are passed through a linear layer that projects each frame to a vocabulary-sized logits vector. This produces per-frame class probabilities aligned with gloss tokens.

5- CTC Loss & Decoding

We train the model using Connectionist Temporal Classification (CTC), which aligns variable-length pose sequences with gloss sequences without requiring frame-level annotations.

During training, CTC computes the probability of all valid alignments between frames and gloss labels.

During inference, we apply greedy CTC decoding and collapse repeated predictions and blanks.

Decoding uses the true (unpadded) sequence length to avoid spurious outputs.

6- Evaluation Metric

Performance is measured using Word Error Rate (WER) computed from:

Substitutions

Insertions

Deletions

between predicted and reference gloss sequences.

**Training Output**



```
Epoch 01: Train Loss=15.2282 | Dev WER=1.0000
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 02: Train Loss=5.9133 | Dev WER=1.0000
Epoch 03: Train Loss=5.7866 | Dev WER=1.0000
Epoch 04: Train Loss=5.7099 | Dev WER=0.9194
Epoch 05: Train Loss=5.6322 | Dev WER=0.9194
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 06: Train Loss=5.5627 | Dev WER=0.9194
Epoch 07: Train Loss=5.5001 | Dev WER=0.9192
Epoch 08: Train Loss=5.4408 | Dev WER=0.9137
Epoch 09: Train Loss=5.3751 | Dev WER=0.9093
Epoch 10: Train Loss=5.2859 | Dev WER=0.9091
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 11: Train Loss=5.1903 | Dev WER=0.9045
Epoch 12: Train Loss=5.0877 | Dev WER=0.9007
Epoch 13: Train Loss=4.9704 | Dev WER=0.8966
Epoch 14: Train Loss=4.8252 | Dev WER=0.8909
Epoch 15: Train Loss=4.6854 | Dev WER=0.8920
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  []
------------------------------------------------------------
Epoch 16: Train Loss=4.5039 | Dev WER=0.8768
Epoch 17: Train Loss=4.3147 | Dev WER=0.8579
Epoch 18: Train Loss=4.1153 | Dev WER=0.8386
Epoch 19: Train Loss=3.8902 | Dev WER=0.8307
Epoch 20: Train Loss=3.6568 | Dev WER=0.8155
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'هو']
------------------------------------------------------------
Epoch 21: Train Loss=3.4062 | Dev WER=0.7910
Epoch 22: Train Loss=3.1493 | Dev WER=0.7743
Epoch 23: Train Loss=2.8965 | Dev WER=0.7589
Epoch 24: Train Loss=2.6457 | Dev WER=0.7433
Epoch 25: Train Loss=2.3835 | Dev WER=0.7253
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال']
------------------------------------------------------------
Epoch 26: Train Loss=2.1560 | Dev WER=0.6930
Epoch 27: Train Loss=1.9152 | Dev WER=0.6680
Epoch 28: Train Loss=1.7132 | Dev WER=0.6629
Epoch 29: Train Loss=1.4992 | Dev WER=0.6390
Epoch 30: Train Loss=1.3316 | Dev WER=0.6269
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'جميل']
------------------------------------------------------------
Epoch 31: Train Loss=1.1673 | Dev WER=0.6265
Epoch 32: Train Loss=1.0291 | Dev WER=0.6296
Epoch 33: Train Loss=0.9001 | Dev WER=0.5955
Epoch 34: Train Loss=0.7966 | Dev WER=0.5797
Epoch 35: Train Loss=0.7155 | Dev WER=0.5766
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'جميل']
------------------------------------------------------------
Epoch 36: Train Loss=0.6173 | Dev WER=0.5722
Epoch 37: Train Loss=0.5612 | Dev WER=0.5558
Epoch 38: Train Loss=0.5053 | Dev WER=0.5606
Epoch 39: Train Loss=0.4416 | Dev WER=0.5501
Epoch 40: Train Loss=0.4002 | Dev WER=0.5698
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف', 'جميل']
------------------------------------------------------------
Epoch 41: Train Loss=0.3737 | Dev WER=0.5408
Epoch 42: Train Loss=0.3451 | Dev WER=0.5455
Epoch 43: Train Loss=0.3202 | Dev WER=0.5222
Epoch 44: Train Loss=0.2933 | Dev WER=0.5402
Epoch 45: Train Loss=0.2814 | Dev WER=0.5246
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['استفهام', 'جميل']
------------------------------------------------------------
Epoch 46: Train Loss=0.2578 | Dev WER=0.5384
Epoch 47: Train Loss=0.2522 | Dev WER=0.5369
Epoch 48: Train Loss=0.2175 | Dev WER=0.5136
Epoch 49: Train Loss=0.2112 | Dev WER=0.5004
Epoch 50: Train Loss=0.2048 | Dev WER=0.5202
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف']
------------------------------------------------------------
Epoch 51: Train Loss=0.2106 | Dev WER=0.5075
Epoch 52: Train Loss=0.1856 | Dev WER=0.4985
Epoch 53: Train Loss=0.1712 | Dev WER=0.4919
Epoch 54: Train Loss=0.1605 | Dev WER=0.5026
Epoch 55: Train Loss=0.1510 | Dev WER=0.5077
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'هو']
------------------------------------------------------------
Epoch 56: Train Loss=0.1693 | Dev WER=0.5059
Epoch 57: Train Loss=0.1443 | Dev WER=0.5026
Epoch 58: Train Loss=0.1393 | Dev WER=0.5108
Epoch 59: Train Loss=0.1372 | Dev WER=0.4939
Epoch 60: Train Loss=0.1362 | Dev WER=0.4978
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف']
------------------------------------------------------------
Epoch 61: Train Loss=0.1084 | Dev WER=0.4868
Epoch 62: Train Loss=0.1242 | Dev WER=0.5125
Epoch 63: Train Loss=0.1356 | Dev WER=0.4864
Epoch 64: Train Loss=0.1230 | Dev WER=0.4785
Epoch 65: Train Loss=0.0979 | Dev WER=0.4980
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'هو']
------------------------------------------------------------
Epoch 66: Train Loss=0.1119 | Dev WER=0.4910
Epoch 67: Train Loss=0.1303 | Dev WER=0.4838
Epoch 68: Train Loss=0.1086 | Dev WER=0.4625
Epoch 69: Train Loss=0.0721 | Dev WER=0.4614
Epoch 70: Train Loss=0.0819 | Dev WER=0.4717
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف']
------------------------------------------------------------
Epoch 71: Train Loss=0.0813 | Dev WER=0.4664
Epoch 72: Train Loss=0.0868 | Dev WER=0.5261
Epoch 73: Train Loss=0.1210 | Dev WER=0.4855
Epoch 74: Train Loss=0.0797 | Dev WER=0.4881
Epoch 75: Train Loss=0.0746 | Dev WER=0.5011
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف', 'هو']
------------------------------------------------------------
Epoch 76: Train Loss=0.0669 | Dev WER=0.4750
Epoch 77: Train Loss=0.0746 | Dev WER=0.4526
Epoch 78: Train Loss=0.0910 | Dev WER=0.4679
Epoch 79: Train Loss=0.0680 | Dev WER=0.4495
Epoch 80: Train Loss=0.1043 | Dev WER=0.4537
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'جميل', 'رحمه']
------------------------------------------------------------
Epoch 81: Train Loss=0.0678 | Dev WER=0.4978
Epoch 82: Train Loss=0.0637 | Dev WER=0.4603
Epoch 83: Train Loss=0.0548 | Dev WER=0.4607
Epoch 84: Train Loss=0.0669 | Dev WER=0.4761
Epoch 85: Train Loss=0.0746 | Dev WER=0.4679
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'اسف', 'هو']
------------------------------------------------------------
Epoch 86: Train Loss=0.0736 | Dev WER=0.4552
Epoch 87: Train Loss=0.0533 | Dev WER=0.4605
Epoch 88: Train Loss=0.0513 | Dev WER=0.4736
Epoch 89: Train Loss=0.0588 | Dev WER=0.4545
Epoch 90: Train Loss=0.0683 | Dev WER=0.4556
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'مال']
------------------------------------------------------------
Epoch 91: Train Loss=0.0678 | Dev WER=0.4466
Epoch 92: Train Loss=0.0831 | Dev WER=0.4671
Epoch 93: Train Loss=0.0967 | Dev WER=0.4609
Epoch 94: Train Loss=0.0536 | Dev WER=0.4326
Epoch 95: Train Loss=0.0315 | Dev WER=0.4256
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['سوال', 'انا']
------------------------------------------------------------
Epoch 96: Train Loss=0.0377 | Dev WER=0.4350
Epoch 97: Train Loss=0.0369 | Dev WER=0.4504
Epoch 98: Train Loss=0.0508 | Dev WER=0.4660
Epoch 99: Train Loss=0.0651 | Dev WER=0.4666
Epoch 100: Train Loss=0.0623 | Dev WER=0.4473
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['هو', 'رحمه']
------------------------------------------------------------

**Test Results**

Final Dev WER = 0.4473

**Error analysis**

1. Substitution Errors (Most Common)

The most frequent mistake in late epochs is substituting the correct gloss with a semantically plausible but incorrect gloss. Examples include:

GT: سوال هو

Pred: سوال اسف

Pred: انا جميل رحمه

Pred: سوال انا

Pred: سؤال اسف هو

Pred: استفهام جميل

Pred: هو رحمه


These errors typically occur because:

Some glosses have similar motion signatures in the pose data.

The model learns semantic clusters (e.g., "اسف", "جميل", "رحمه") and sometimes chooses a gloss from the wrong cluster.

CTC tends to collapse nearby frames into one alignment, making subtle distinctions difficult.

This type of error dominates final training behavior.

2. Semantic Drift

In some late-epoch predictions, the model produces a fully plausible but entirely wrong gloss sequence, such as:

['انا', 'جميل', 'رحمه']
['انا', 'مال']


This happens when:

The model recognizes the general style of the gesture but not the exact gloss.

Pose-only features lack detail for fine distinctions.

The model relies on learned gloss co-occurrence patterns instead of precise motion cues.


5. End-of-Sequence Errors

The final gloss in the sequence ("هو") is the most frequently mispredicted. Typical replacement patterns include:

هو - جميل

هو - اسف

هو - رحمه

هو - انا


Reasons:

The end of the video often contains weak or fading motion.

Attention typically focuses more strongly on early frames.

CTC alignment is less stable near the end of the sequence.

This explains why the first gloss is usually correct, while the second gloss is often substituted.